# 14 - SATISFIED, VIOLATED, and REJECTED

## Objetivo
Comparar lado a lado os três desfechos possíveis para um intent que chega
a ser processado pelo planner, deixando claro em qual etapa cada um se
decide: `REJECTED` antes de qualquer simulação, `VIOLATED` depois de uma
simulação real insuficiente, `SATISFIED` depois de uma simulação real
suficiente.

## Conceitos

**SATISFIED**: plano viável, reserva aceita pelo `NetworkManager`, e todas
as condições de `validation.success_conditions` passam contra evidência
`DELIVERY` real.

**VIOLATED**: plano considerado viável **na estimativa** (fidelidade e
memória bastam), a reserva é aceita e a rede realmente roda - mas o
resultado operacional (normalmente `delivered_pairs`, já que o planner v1
não estima throughput, ver `docs/limitations.md`) fica aquém do
declarado. Pelo menos uma condição falha.

**REJECTED**: inviabilidade detectada em `IntentPlanner.plan` **antes**
de qualquer chamada ao SeQUeNCe (`ExecutionPlan.feasible=False`) -
`SequenceExecutor._deploy_plan` transiciona direto para `REJECTED` e
retorna sem nunca chamar `_start_reservation`. Não existe reserva, não
existe `Timeline` rodando para este intent, não existe evidência de
entrega.


## Configuração

Três intents sobre a mesma topologia base `a - r - b`, cada um isolando um desfecho.


In [1]:
from ibqn.demos.topologies import three_node_spec
from ibqn.demos.intents import simple_intent
from ibqn.demos.scenarios import ad_hoc_scenario
from ibqn.experiments.runner import run_scenario

SEED = 0

spec_satisfied = three_node_spec(stop_time_s=0.1)  # default low attenuation (1e-5 dB/m)
intent_satisfied = simple_intent(
    intent_id="case-satisfied", source="a", destination="b",
    min_fidelity=0.65, requested_pairs=10, duration=0.05, start_time=0.02,
)

spec_rejected = three_node_spec(stop_time_s=0.1)
intent_rejected = simple_intent(
    intent_id="case-rejected", source="a", destination="b",
    min_fidelity=0.999, requested_pairs=10, duration=0.05, start_time=0.02,  # above what even purification can reach
)

# High attenuation (0.025 dB/m): fidelity/memory are both fine at planning
# time (attenuation only affects photon-loss probability, not the fidelity
# of a heralded pair or of a swap - docs/physical_model.md), but the real run
# cannot deliver requested_pairs within the window - a genuine, observed
# violation the v1 planner cannot predict in advance.
spec_violated = three_node_spec(stop_time_s=0.2, attenuation_db_per_m=0.025)
intent_violated = simple_intent(
    intent_id="case-violated", source="a", destination="b",
    min_fidelity=0.6, requested_pairs=10, duration=0.1, start_time=0.02,
)


2026-10-02 15:03:49,757 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:49,763 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:49,764 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:49,766 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:49,768 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:49,769 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:49,772 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:49,773 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:49,774 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


## Execução


In [2]:
result_satisfied = run_scenario(ad_hoc_scenario(spec_satisfied, [intent_satisfied], seed=SEED), seed=SEED).get(intent_satisfied.id)
result_rejected = run_scenario(ad_hoc_scenario(spec_rejected, [intent_rejected], seed=SEED), seed=SEED).get(intent_rejected.id)
result_violated = run_scenario(ad_hoc_scenario(spec_violated, [intent_violated], seed=SEED), seed=SEED).get(intent_violated.id)

for label, r in [("SATISFIED", result_satisfied), ("REJECTED", result_rejected), ("VIOLATED", result_violated)]:
    print(f"{label}: final_status={r.final_status.value}, plan.feasible={r.plan.feasible}")


2026-10-02 15:03:49,802 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - topology built: 3 routers, 2 quantum links, formalism=bell_diagonal, seed=0


2026-10-02 15:03:49,805 INFO     ibqn.ibqn.planning.planner intent_id=case-satisfied - planned route a->r->b, estimated_fidelity=0.7300, purification=False


2026-10-02 15:03:49,807 INFO     ibqn.ibqn.intent.repository intent_id=case-satisfied - intent registered


2026-10-02 15:03:49,809 INFO     ibqn.ibqn.intent.repository intent_id=case-satisfied - intent transitioned to VALIDATED (schema validated)


2026-10-02 15:03:49,810 INFO     ibqn.ibqn.intent.repository intent_id=case-satisfied - intent transitioned to PLANNING (invoking IntentPlanner)


...

2026-10-02 15:03:52,896 INFO     ibqn.ibqn.experiments.runner intent_id=- - running scenario 'ad-hoc-demo', seed=0, 1 intent(s)


2026-10-02 15:03:52,897 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - running simulation, stop_time=0.200s


2026-10-02 15:03:52,899 INFO     ibqn.ibqn.intent.repository intent_id=case-violated - intent transitioned to ACTIVE (reservation approved)


2026-10-02 15:03:55,956 INFO     ibqn.ibqn.intent.repository intent_id=case-violated - intent transitioned to VIOLATED (delivered_pairs >= 10.0 (observed=2.0) -> FAIL)


SATISFIED: final_status=SATISFIED, plan.feasible=True
REJECTED: final_status=REJECTED, plan.feasible=False
VIOLATED: final_status=VIOLATED, plan.feasible=True


## Resultados


In [3]:
import pandas as pd

def summarize(label, result):
    delivered = None
    avg_fidelity = None
    violations = "(none)"
    if result.evaluation is not None:
        for c in result.evaluation.condition_results:
            if c.metric == "delivered_pairs":
                delivered = c.observed
            elif c.metric == "average_fidelity":
                avg_fidelity = c.observed
        violations = "; ".join(result.evaluation.violations) or "(none)"
    return {
        "Case": label,
        "Planner result": "feasible" if result.plan.feasible else f"infeasible ({result.plan.infeasibility_reason})",
        "Network executed": "yes (reservation accepted, Timeline ran)" if result.plan.feasible else "no (rejected before touching SeQUeNCe)",
        "Delivered pairs": delivered,
        "Avg. fidelity": round(avg_fidelity, 4) if avg_fidelity is not None else None,
        "Violations": violations,
        "Final status": result.final_status.value,
    }

summary_df = pd.DataFrame([
    summarize("SATISFIED", result_satisfied),
    summarize("REJECTED", result_rejected),
    summarize("VIOLATED", result_violated),
])
summary_df


,Case,Planner result,Network executed,Delivered pairs,Avg. fidelity,Violations,Final status
0,SATISFIED,feasible,"yes (reservation accepted, Timeline ran)",319.0,0.7287,(none),SATISFIED
1,REJECTED,infeasible (no candidate route is feasible: a-...,no (rejected before touching SeQUeNCe),NaN,NaN,(none),REJECTED
2,VIOLATED,feasible,"yes (reservation accepted, Timeline ran)",2.0,0.6974,delivered_pairs >= 10.0 (observed=2.0) -> FAIL,VIOLATED


In [4]:
from ibqn.intent.models import IntentStatus

assert result_satisfied.final_status == IntentStatus.SATISFIED
assert result_violated.final_status == IntentStatus.VIOLATED
assert result_rejected.final_status == IntentStatus.REJECTED
print("All three outcomes confirmed with their expected final_status.")


All three outcomes confirmed with their expected final_status.


In [5]:
assert result_rejected.evaluation is None, "a REJECTED intent has no assurance evaluation - it never reached ACTIVE"
assert result_rejected.satisfied is False
print("Confirmed: REJECTED carries no delivery evidence at all (no evaluation object exists).")


Confirmed: REJECTED carries no delivery evidence at all (no evaluation object exists).


## Interpretação
O caso `REJECTED` nunca aparece na tabela com um número de pares
entregues: `result.evaluation` é `None` porque `SequenceExecutor` nunca
chamou `_start_reservation` para esse intent - o `IntentPlanner` já sabia,
antes de qualquer `Timeline`, que `min_fidelity=0.999` excede o que mesmo
a purificação pode alcançar analiticamente. Já o caso `VIOLATED` reservou
recursos de verdade, rodou a simulação inteira, e só então descobriu (via
evidência `DELIVERY`) que a alta atenuação do link tornou a geração de
entrelaçamento lenta demais para entregar os pares pedidos na janela - uma
limitação que o planner v1 não consegue prever de antemão, porque ele não
estima throughput (`docs/limitations.md`).

## Limitações
- A calibração do caso `VIOLATED` depende dos **nomes** dos nós, não só
  da física. No SeQUeNCe, o papel *primário* de cada enlace na geração
  cabe ao nó de nome lexicograficamente maior, e uma tentativa leva 4
  atrasos clássicos quando quem pede o pareamento (o nó anterior na rota)
  não é o primário e 5 quando é (`docs/generation_model_audit.md`). Em
  `a - r - b` o enlace `r - b` cai no caso lento (`r` > `b`); com nós
  `r1`/`r2`/`r3` nenhum enlace cai. Isso não compromete a
  reprodutibilidade (a mesma seed + mesma topologia sempre reproduz o
  mesmo resultado), mas valores de calibração não são transferíveis entre
  topologias com nomes diferentes sem reverificação empírica -
  exatamente o que foi feito aqui.
- O planner v1 não estima `delivered_pairs`/throughput - por isso
  `VIOLATED` só pode ser descoberto observando a simulação real, nunca
  antecipado no plano. Os planejadores cientes de recursos
  (`ibqn.planning.planners`, níveis L2-RB/L3-RB) estimam a entrega e
  rejeitam no planejamento um intent cuja meta declarada
  (`min_delivered_pairs`) não cabe na janela
  (`docs/generation_model_audit.md`).

## Conclusão
Os três desfechos terminais mais comuns do lifecycle demonstrados lado a
lado, com a distinção central explícita: `REJECTED` decide-se sem tocar o
SeQUeNCe; `SATISFIED`/`VIOLATED` só se decidem depois de uma simulação
real, a partir de evidência `DELIVERY` genuína.
